# Preprocessing Notebook

This notebook prepares analysis-ready tables from the raw YouTube collection outputs.

**How to use:**
- Update `PERSON` below to switch datasets (e.g., `bodeoum_tv` or `mimiminu`).
- Rerun all cells to produce outputs under `02-preprocessing/outputs/<person>/`.


## 0) Imports and configuration
This cell sets the dataset name and all input/output paths.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm import tqdm

# ---- CONFIG ----
PERSON = "mimiminu"  # change to 'mimiminu' when needed
LANG = "ko"

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "data").exists() and (REPO_ROOT / "02-preprocessing").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = REPO_ROOT / "data"
INPUT_CSV = DATA_ROOT / "input" / f"{PERSON}.csv"
YOUTUBE_JSONL = DATA_ROOT / PERSON / f"{PERSON}_youtube.jsonl"
COMMENTS_DIR = DATA_ROOT / PERSON / "comments"
TRANSCRIPTS_DIR = DATA_ROOT / PERSON / "transcripts"

OUTPUT_DIR = REPO_ROOT / "02-preprocessing" / "outputs" / PERSON
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using PERSON=", PERSON)
print("Input CSV:", INPUT_CSV)
print("YouTube JSONL:", YOUTUBE_JSONL)
print("Comments dir:", COMMENTS_DIR)
print("Transcripts dir:", TRANSCRIPTS_DIR)
print("Output dir:", OUTPUT_DIR)


Using PERSON= mimiminu
Input CSV: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/input/mimiminu.csv
YouTube JSONL: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/mimiminu/mimiminu_youtube.jsonl
Comments dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/mimiminu/comments
Transcripts dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/mimiminu/transcripts
Output dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/02-preprocessing/outputs/mimiminu


## 1) Load window map (CSV)
This builds a `slot -> video_url -> video_id` mapping based on the CSV order.


In [2]:
def extract_video_id(url: str) -> str | None:
    if "watch?v=" in url:
        return url.split("watch?v=", 1)[1].split("&", 1)[0]
    if "youtu.be/" in url:
        return url.split("youtu.be/", 1)[1].split("?", 1)[0]
    return None

window_df = pd.read_csv(INPUT_CSV)
window_df["video_id"] = window_df["video"].astype(str).apply(extract_video_id)

slot_rows = []
for window_label in ("pre", "cancel", "post"):
    rows = window_df[window_df["window"].str.lower() == window_label]
    for i, (_, row) in enumerate(rows.iterrows(), start=1):
        slot = f"{window_label}-{i}"
        slot_rows.append({
            "slot": slot,
            "window": window_label,
            "video": row["video"],
            "video_id": row["video_id"],
        })

slot_df = pd.DataFrame(slot_rows)
slot_df.head()


,slot,window,video,video_id
0,pre-1,pre,https://www.youtube.com/watch?v=GCDooyBOfDU,GCDooyBOfDU
1,pre-2,pre,https://www.youtube.com/watch?v=K04O8NNXtIk,K04O8NNXtIk
2,pre-3,pre,https://www.youtube.com/watch?v=_O8QOqa-RyQ,_O8QOqa-RyQ
3,pre-4,pre,https://www.youtube.com/watch?v=188m_yF45Xw,188m_yF45Xw
4,pre-5,pre,https://www.youtube.com/watch?v=C6Jc8bcdcws,C6Jc8bcdcws


## 2) Extract video metadata (`videos.csv`)
Parses `<person>_youtube.jsonl` and keeps records where `record_type == "video"`.


In [3]:
video_rows = []
with open(YOUTUBE_JSONL, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        if record.get("record_type") != "video":
            continue
        video_rows.append({
            "video_id": record.get("video_id"),
            "title": record.get("title"),
            "video_url": record.get("video_url"),
            "create_time": record.get("create_time"),
            "view_count": record.get("view_count"),
            "like_count": record.get("like_count"),
            "comment_count": record.get("comment_count"),
            "window_label": record.get("window_label"),
        })

videos_df = pd.DataFrame(video_rows)
videos_df.to_csv(OUTPUT_DIR / "videos.csv", index=False)
videos_df.head()


,video_id,title,video_url,create_time,view_count,like_count,comment_count,window_label
0,GCDooyBOfDU,“학생들 인스타그램 염탐하세요?” 서울대 고려대 연세대 출신 선생님들끼리 기싸움했습니다,https://www.youtube.com/watch?v=GCDooyBOfDU,2025-11-22T00:00:00+00:00,226159,2913,297,pre
1,K04O8NNXtIk,"""가천대학교에서는 워터밤이 열려요"" 가천대학교에 가서 대학생들 학점 판독하고 왔습니...",https://www.youtube.com/watch?v=K04O8NNXtIk,2025-11-20T00:00:00+00:00,406057,4040,470,pre
2,_O8QOqa-RyQ,"""찍기 특강 믿었는데 망했어요"" 불수능을 넘어 잔인했던 2026수능 영어 리뷰",https://www.youtube.com/watch?v=_O8QOqa-RyQ,2025-11-18T00:00:00+00:00,745942,7209,1400,pre
3,188m_yF45Xw,"""킬러 문항 배제? 됐고, 킬러 부활했어요"" 모의고사와 느낌이 달랐던 2026수능 ...",https://www.youtube.com/watch?v=188m_yF45Xw,2025-11-17T00:00:00+00:00,434286,3625,732,pre
4,C6Jc8bcdcws,"""모의고사 때보다 독서 왜 이렇게 어려워요?"" 체감 난도가 미쳐 날뛰었던 2026수...",https://www.youtube.com/watch?v=C6Jc8bcdcws,2025-11-16T00:00:00+00:00,630789,6850,1600,pre


## 3) Flatten comments (`comments.csv`)
Reads `comments/*.info.json` and extracts comment fields + window slot.


In [4]:
comment_rows = []
comment_files = sorted(COMMENTS_DIR.glob("*.info.json"))

for path in tqdm(comment_files, desc="comments"):
    slot = path.stem.replace(".info", "")
    window = slot.split("-")[0]
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    video_id = payload.get("id")
    comments = payload.get("comments", [])
    for c in comments:
        comment_rows.append({
            "comment_id": c.get("id"),
            "parent_id": c.get("parent"),
            "author": c.get("author"),
            "author_id": c.get("author_id"),
            "timestamp": c.get("timestamp"),
            "like_count": c.get("like_count"),
            "text": c.get("text"),
            "video_id": video_id,
            "window": window,
            "slot": slot,
        })

comments_df = pd.DataFrame(comment_rows)
comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
comments_df.head()


comments:   0%|                                                  | 0/21 [00:00<?, ?it/s]

comments:  52%|████████████████████▉                   | 11/21 [00:00<00:00, 106.35it/s]

comments: 100%|████████████████████████████████████████| 21/21 [00:00<00:00, 110.16it/s]

,comment_id,parent_id,author,author_id,timestamp,like_count,text,video_id,window,slot
0,UgzUrtgZAy13tMIbOqZ4AaABAg,root,@mimiminu_official,UCyNHdF4hMxHUFSsPbY24p6A,1766188800,289,"법무법인 '트라이원스' 측에서 본 영상의 취지에 깊이 공감해주시어, 감사하게도 이번...",txst-JeT4Qk,cancel,cancel-1
1,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQ8_RZtZdoD,UgzUrtgZAy13tMIbOqZ4AaABAg,@연우-e2j,UC3PySuxHTUPnq6LrfelpMSA,1766188800,27,미미미누 대응 최고면 개추,txst-JeT4Qk,cancel,cancel-1
2,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQ8n6cf5Mm4,UgzUrtgZAy13tMIbOqZ4AaABAg,@또도-z1c,UCiu-V3K7sYMvmmZdN6k3PdQ,1766188800,12,법률 지원을 받는다니 너무 다행이네요😢,txst-JeT4Qk,cancel,cancel-1
3,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQ99iSwIr6G,UgzUrtgZAy13tMIbOqZ4AaABAg,@김광석-w9i,UC3RFGq5bhhplaJAP7VV-qSQ,1766188800,6,구글폼으로만 접수받는다고합니다,txst-JeT4Qk,cancel,cancel-1
4,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQBJCj-uUO1,UgzUrtgZAy13tMIbOqZ4AaABAg,@rucyjw,UCyNnjWS3anKmSsTJEzo1Nxw,1766188800,1,"피해자들도, 민우씨도 힘내요.\n이 대표라는 사람은 어디 숨었냐.. 🤨",txst-JeT4Qk,cancel,cancel-1


## 4) Build reply edges (`edges.csv`)
Creates reply graph edges from `comments.csv`.


In [5]:
edges_df = comments_df[comments_df["parent_id"].notna() & (comments_df["parent_id"] != "root")].copy()
edges_df = edges_df.rename(columns={
    "comment_id": "source_comment_id",
    "parent_id": "target_comment_id",
})
edges_df = edges_df[[
    "source_comment_id",
    "target_comment_id",
    "author_id",
    "video_id",
    "window",
    "slot",
]]
edges_df.to_csv(OUTPUT_DIR / "edges.csv", index=False)
edges_df.head()


,source_comment_id,target_comment_id,author_id,video_id,window,slot
1,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQ8_RZtZdoD,UgzUrtgZAy13tMIbOqZ4AaABAg,UC3PySuxHTUPnq6LrfelpMSA,txst-JeT4Qk,cancel,cancel-1
2,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQ8n6cf5Mm4,UgzUrtgZAy13tMIbOqZ4AaABAg,UCiu-V3K7sYMvmmZdN6k3PdQ,txst-JeT4Qk,cancel,cancel-1
3,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQ99iSwIr6G,UgzUrtgZAy13tMIbOqZ4AaABAg,UC3RFGq5bhhplaJAP7VV-qSQ,txst-JeT4Qk,cancel,cancel-1
4,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQBJCj-uUO1,UgzUrtgZAy13tMIbOqZ4AaABAg,UCyNnjWS3anKmSsTJEzo1Nxw,txst-JeT4Qk,cancel,cancel-1
5,UgzUrtgZAy13tMIbOqZ4AaABAg.AQ8_GY9Uka2AQBtpFt1fQ5,UgzUrtgZAy13tMIbOqZ4AaABAg,UCN-cccbN4-7VcobDZncUpqw,txst-JeT4Qk,cancel,cancel-1


## 5) Parse transcripts (`transcripts.csv`)
Reads `transcripts/*.ko.txt` and attaches window/slot.


In [6]:
transcript_rows = []
for path in sorted(TRANSCRIPTS_DIR.glob(f"*.{LANG}.txt")):
    name = path.name
    # expected: pre-1.ko.txt / cancel-1.ko.txt / post-1.ko.txt
    slot = name.split(".")[0]
    window = slot.split("-")[0]
    text = path.read_text(encoding="utf-8")
    transcript_rows.append({
        "slot": slot,
        "window": window,
        "transcript_text": text,
    })

transcripts_df = pd.DataFrame(transcript_rows)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)
transcripts_df.head()


,slot,window,transcript_text
0,cancel-1,cancel,안녕하세요. 미이미누입니다.\n제 채널에서 광고를 진행했던 파트타임 스터디 서비스가...


## 6) Text cleaning
Enhanced preprocessing for informal Korean social media text:
1. Remove zero-width characters and HTML entities
2. Strip embedded URLs
3. Preserve Hangul (jamo + syllables), emojis, punctuation, Latin, digits
4. Normalize whitespace
5. Apply spacing normalization for Korean word boundaries
6. Deduplicate spam (same text, same author, 60s window)

In [7]:
import re
import kss
from datetime import datetime

url_re = re.compile(r"https?://\S+")
whitespace_re = re.compile(r"\s+")
zw_re = re.compile(r"[\u200b-\u200f\ufeff\u180b-\u180f]")
html_entity_re = re.compile(r"&[a-zA-Z]+;|&#x[a-fA-F0-9]+;|#[0-9]+;")
char_whitelist_re = re.compile(r"[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7FFa-zA-Z0-9.,!?~()\u2000-\u206F\u2100-\u214F\u2190-\u21FF\u2200-\u22FF\u2300-\u23FF\u2500-\u257F\u2600-\u26FF\u2700-\u27BF\U0001F300-\U0001F6FF\U0001F700-\U0001F9FF\U0001FA00-\U0001FAFF]+")

def clean_text(text: str | None) -> str:
    if not text:
        return ""
    text = zw_re.sub("", text)
    text = html_entity_re.sub("", text)
    text = url_re.sub("", text)
    text = " ".join(char_whitelist_re.findall(text))
    text = whitespace_re.sub(" ", text).strip()
    return text

def normalize_korean_spacing(text: str) -> str:
    if not text or len(text.strip()) < 2:
        return text
    try:
        return " ".join(kss.split_sentences(text, strip=True))
    except:
        return text

def deduplicate_spam(df: pd.DataFrame, time_window: int = 60) -> pd.DataFrame:
    df = df.copy()
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df = df.sort_values(["author_id", "timestamp"])
    mask = ~(
        (df["text_clean"] == df["text_clean"].shift(1))
        & (df["author_id"] == df["author_id"].shift(1))
        & ((df["timestamp"] - df["timestamp"].shift(1)) <= time_window)
    )
    return df[mask]

print("Cleaning comments...")
comments_df["text_clean"] = comments_df["text"].astype(str).apply(clean_text)
print("Cleaning transcripts...")
transcripts_df["text_clean"] = transcripts_df["transcript_text"].astype(str).apply(clean_text)

comments_df = comments_df[comments_df["text_clean"].str.strip().astype(bool)]
print(f"Comments before dedup: {len(comments_df)}")
comments_df = deduplicate_spam(comments_df)
print(f"Comments after dedup: {len(comments_df)}")

print("Normalizing Korean spacing for comments...")
comments_df["text_clean"] = comments_df["text_clean"].apply(normalize_korean_spacing)

comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)

print("\nSample cleaned comments:")
comments_df[["text", "text_clean"]].head()


In [7]:
import re

url_re = re.compile(r"https?://\S+")
whitespace_re = re.compile(r"\s+")

def clean_text(text: str | None) -> str:
    if not text:
        return ""
    text = url_re.sub("", text)
    text = whitespace_re.sub(" ", text).strip()
    return text

comments_df["text_clean"] = comments_df["text"].astype(str).apply(clean_text)
transcripts_df["text_clean"] = transcripts_df["transcript_text"].astype(str).apply(clean_text)

comments_df = comments_df[comments_df["text_clean"].str.strip().astype(bool)]

comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)

comments_df[["text", "text_clean"]].head()


,text,text_clean
0,"법무법인 '트라이원스' 측에서 본 영상의 취지에 깊이 공감해주시어, 감사하게도 이번...","법무법인 '트라이원스' 측에서 본 영상의 취지에 깊이 공감해주시어, 감사하게도 이번..."
1,미미미누 대응 최고면 개추,미미미누 대응 최고면 개추
2,법률 지원을 받는다니 너무 다행이네요😢,법률 지원을 받는다니 너무 다행이네요😢
3,구글폼으로만 접수받는다고합니다,구글폼으로만 접수받는다고합니다
4,"피해자들도, 민우씨도 힘내요.\n이 대표라는 사람은 어디 숨었냐.. 🤨","피해자들도, 민우씨도 힘내요. 이 대표라는 사람은 어디 숨었냐.. 🤨"


## 7) Summary
Quick sanity checks on row counts and output paths.


In [8]:
print("videos.csv", len(videos_df))
print("comments.csv", len(comments_df))
print("edges.csv", len(edges_df))
print("transcripts.csv", len(transcripts_df))
print("Output dir:", OUTPUT_DIR)


videos.csv 21
comments.csv 16792
edges.csv 7110
transcripts.csv 1
Output dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/02-preprocessing/outputs/mimiminu
